# Stage 0–1 walkthrough — bimanual task & role generation

Runs the **real** pipeline on one object, step by step, by **importing the canonical scripts**
(`pipeline/stage0_filter_components.py` + `pipeline/stage1_task_role_assemble.py`). No inlined
prompt copies, so this notebook always matches exactly what the batch scripts produce.
Model = **gemma-4-26B-A4B-it (no-thinking)** via vLLM.

## 0 · Setup — import the canonical stage scripts

In [1]:
import os, sys, json
os.environ.setdefault("CUDA_DEVICE_ORDER", "PCI_BUS_ID")
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "3")
os.environ.setdefault("GEMMA_BACKEND", "vllm")
os.environ.setdefault("GEMMA_MODEL_ID", "google/gemma-4-26B-A4B-it")
os.environ.setdefault("GEMMA_NO_THINKING", "1")
os.environ.setdefault("VLLM_USE_FLASHINFER_SAMPLER", "0")
os.environ.setdefault("VLLM_WORKER_MULTIPROC_METHOD", "spawn")

REPO = "/home/michaellee/mclee/affogato"; DG = REPO + "/data_generation"
sys.path.insert(0, REPO + "/bimanual_annotation")
sys.path.insert(0, DG + "/pipeline")
os.chdir(REPO)                                   # so prompt/ + dataset/ relative paths resolve

import stage0_filter_components as S0            # filter + component extraction  (canonical)
import stage1_task_role_assemble as S1           # task proposal + role decomposition (canonical, v9 prompts)
from get_component import ComponentExtractor
print("imported canonical stage0 / stage1 — every prompt below comes straight from these scripts")

imported canonical stage0 / stage1 — every prompt below comes straight from these scripts


## 1 · Load the model (gemma-4-26B-A4B, no-thinking, vLLM)

In [3]:
ce = ComponentExtractor()        # loads Gemma (GEMMA_MODEL_ID) + the stage-0 filter/parse prompts
model = ce.model
print("model:", os.environ["GEMMA_MODEL_ID"], "| no_thinking:", getattr(model, "no_thinking", None))

Loading Gemma (vLLM)... (google/gemma-4-26B-A4B-it)  max_model_len=32768 soft_tokens=560/1120 enforce_eager=False
INFO 06-08 13:52:55 [api_utils.py:273] non-default args: {'trust_remote_code': True, 'max_model_len': 32768, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'hf_overrides': {'vision_config': {'default_output_length': 1120}, 'vision_soft_tokens_per_image': 1120}, 'limit_mm_per_prompt': {'image': 8}, 'mm_processor_kwargs': {'max_soft_tokens': 560}, 'model': 'google/gemma-4-26B-A4B-it'}
INFO 06-08 13:52:56 [model.py:611] Resolved architecture: Gemma4ForConditionalGeneration
INFO 06-08 13:52:56 [model.py:1745] Using max model len 32768
INFO 06-08 13:52:56 [config.py:100] Gemma4 model has heterogeneous head dimensions (head_dim=256, global_head_dim=512). Forcing TRITON_ATTN backend to prevent mixed-backend numerical divergence.
INFO 06-08 13:52:56 [kernel.py:270] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rm

(EngineCore pid=2935039) /home/michaellee/miniconda3/envs/gemma4/lib/python3.12/site-packages/paddle/utils/cpp_extension/extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
(EngineCore pid=2935039)   warnings.warn(warning_message)


(EngineCore pid=2935039) INFO 06-08 13:53:17 [core.py:113] Initializing a V1 LLM engine (v0.22.1rc1.dev237+gfa27d4e9c) with config: model='google/gemma-4-26B-A4B-it', speculative_config=None, tokenizer='google/gemma-4-26B-A4B-it', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=32768, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=None, quantization_config=None, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_hidden_metrics_for_

(EngineCore pid=2935039) <frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
(EngineCore pid=2935039) <frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.nvrtc module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.nvrtc module instead.


(EngineCore pid=2935039) INFO 06-08 13:53:29 [weight_utils.py:922] Filesystem type for checkpoints: EXT4. Checkpoint size: 48.07 GiB. Available RAM: 229.92 GiB.
(EngineCore pid=2935039) INFO 06-08 13:53:29 [weight_utils.py:945] Auto-prefetch is disabled because the filesystem (EXT4) is not a recognized network FS (NFS/Lustre). If you want to force prefetching, start vLLM with --safetensors-load-strategy=prefetch.


Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  50% Completed | 1/2 [01:39<01:39, 99.05s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [01:44<00:00, 43.74s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [01:44<00:00, 52.03s/it]
(EngineCore pid=2935039) 


(EngineCore pid=2935039) INFO 06-08 13:55:13 [default_loader.py:397] Loading weights took 104.13 seconds
(EngineCore pid=2935039) INFO 06-08 13:55:13 [unquantized.py:347] Using MoEPrepareAndFinalizeNoDPEPModular
(EngineCore pid=2935039) INFO 06-08 13:55:14 [gpu_model_runner.py:5187] Model loading took 48.54 GiB memory and 107.767733 seconds
(EngineCore pid=2935039) INFO 06-08 13:55:15 [gpu_model_runner.py:6200] Encoder cache will be initialized with a budget of 16384 tokens, and profiled with 6 video items of the maximum feature size.
(EngineCore pid=2935039) INFO 06-08 13:56:11 [backends.py:1089] Using cache directory: /home/michaellee/.cache/vllm/torch_compile_cache/4ecb0817d0/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=2935039) INFO 06-08 13:56:11 [backends.py:1148] Dynamo bytecode transform time: 5.31 s
(EngineCore pid=2935039) INFO 06-08 13:56:13 [backends.py:292] Directly load the compiled graph(s) for compile range (1, 16384) from the cache, took 2.065 s
(EngineCo

(EngineCore pid=2935039) 2026-06-08 13:56:18,991 - INFO - autotuner.py:622 - flashinfer.jit: [Autotuner]: Autotuning process starts ...
(EngineCore pid=2935039) 2026-06-08 13:56:19,052 - INFO - autotuner.py:641 - flashinfer.jit: [Autotuner]: Autotuning process ends
Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:05<00:00,  9.95it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 51/51 [00:05<00:00,  9.88it/s]


(EngineCore pid=2935039) INFO 06-08 13:56:30 [gpu_model_runner.py:6585] Graph capturing finished in 12 secs, took 0.88 GiB
(EngineCore pid=2935039) INFO 06-08 13:56:30 [gpu_worker.py:632] CUDA graph pool memory: 0.88 GiB (actual), 0.85 GiB (estimated), difference: 0.02 GiB (2.7%).
(EngineCore pid=2935039) INFO 06-08 13:56:30 [jit_monitor.py:54] Kernel JIT monitor activated — Triton JIT compilations during inference will be logged as warnings.
(EngineCore pid=2935039) INFO 06-08 13:56:31 [core.py:306] init engine (profile, create kv cache, warmup model) took 76.58 s (compilation: 7.88 s)
(EngineCore pid=2935039) INFO 06-08 13:56:31 [kernel.py:270] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
Gemma loaded!
model: google/gemma-4-26B-A4B-it | no_thinking: True


## 2 · Pick an object
Any object from the stage-0 kept set (its name + the 40-view gObjaverse render paths).

In [4]:
OBJ_ID = "fc827b6683d8441f9b7c30609b5cc371"      # Cardboard Box (a co-lift pick-up — the v9 case)
src = next(o for o in json.load(open(f"{DG}/outputs/stage0_filtered_redesign.kept.json"))
           if o["object_id"] == OBJ_ID)
name, views = src["object_name"], src["views_used"]
pairs  = S0.safe_prepare_views(views, S0.VIEW_DIR)            # decode + label the renders
clean  = [p for _, p in pairs]
labels = [S0.view_label(views[idx]) for idx, _ in pairs]
print(f"object: {name}  |  {len(clean)} labelled views")

object: Cardboard Box  |  8 labelled views


## 3 · Stage 0 — everyday-filter + component extraction
The *exact* stage-0 prompts: `S0.FILTER_PROMPT` (keep/drop) and the parse prompt (evidence-gated parts —
nothing inferred from the name).

In [5]:
# (a) everyday-object filter -> keep/drop + an example bimanual task
filt = S0.parse_json(model.text_images(
    S0.FILTER_PROMPT["user"].format(object_name=name), clean, S0.FILTER_PROMPT["system"], labels=labels))
print("filter:", {k: filt.get(k) for k in ("keep", "everyday_object", "example_task")})

# (b) component extraction -> the groundable parts
pp = ce.prompts["parse"]
comps = S0.parse_json(model.text_images(
    pp["user"].format(object_name=name), clean, pp["system"], labels=labels)).get("canonical_components", [])
comps = S1.ensure_body_component(name, comps)                # guarantee a rigid 'body' target
print("components:", [c.get("name") for c in comps])

(EngineCore pid=2935039) WARNING 06-08 13:57:26 [jit_monitor.py:103] Triton kernel JIT compilation during inference: _compute_slot_mapping_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.
(EngineCore pid=2935039) WARNING 06-08 13:57:27 [jit_monitor.py:103] Triton kernel JIT compilation during inference: kernel_unified_attention. This causes a latency spike; consider extending warmup to cover this shape/config.
(EngineCore pid=2935039) WARNING 06-08 13:57:27 [jit_monitor.py:103] Triton kernel JIT compilation during inference: fused_moe_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.
filter: {'keep': True, 'everyday_object': True, 'example_task': 'Lifting and carrying the box with both hands to move it to another location.'}
components: ['body']


## 4 · Stage 1 — task proposal
brainstorm two families → rank → assemble; the universal pick-up / move is injected automatically.

In [6]:
tasks = S1.propose_tasks(model, name, comps, clean, labels=labels)
for t in tasks:
    print(f"  ({t['category']:5}) {t['task']}")

  (inter) pick up the Cardboard Box
  (inter) flip the box
  (inter) flip the box to show the shipping label


## 5 · Stage 1 — coordination-aware role decomposition
Each task → `OP_PLAN` (physics) → `GROUND` (two hand-agnostic roles). All the v9 physics lives here:
force-point lift, **co-lift** (grip-less → both hands lift opposite faces), intrinsic gravity-frame contacts.

In [7]:
records = []
for t in tasks:
    dec = S1.decompose_task(model, name, comps, t, clean, labels=labels)
    if not dec:
        print(f"  [dropped] {t['task']}"); continue
    records.append(dec)
    print(f"\n  {dec['task']}   [{dec['coordination']}]")
    for r in dec["roles"]:
        print(f"     {r['id']}: {r['role']:7} {r['target']:14} @ {r['contact_region']}")


  pick up the Cardboard Box   [whole-body]
     A: lift    body           @ the mid-height of one vertical side wall
     B: lift    body           @ the mid-height of the opposite vertical side wall

  flip the box   [whole-body]
     A: rotate  body           @ the side wall near the top edge
     B: support body           @ the opposite side wall near the top edge

  flip the box to show the shipping label   [whole-body]
     A: rotate  body           @ the mid-height of a side face
     B: hold    body           @ the mid-height of the opposite side face


## 6 · The assembled record
= exactly one entry the batch script writes to `outputs/stage1_redesign_26b.json`.

In [9]:
rec = {"object_id": OBJ_ID, "object_name": name, "views_used": views,
       "components": comps, "queries": records}
print(json.dumps(rec["queries"][0], indent=2, ensure_ascii=False))

{
  "task": "pick up the Cardboard Box",
  "query": "How would you pick up the Cardboard Box?",
  "goal": "Cardboard Box resting -> Cardboard Box lifted clear of the surface and held",
  "why_bimanual": "raising it clear and level beats gravity only with a weight-bearing hold; one hand cannot both bear the load and keep it from tipping, so two hands must lift and steady it together, or it tips or slips",
  "category": "inter",
  "mechanism": "none",
  "operation": "{\"size_class\": \"hand-sized\", \"moving_part\": \"none\", \"motion\": \"none\", \"direction_to_goal\": \"upward from surface\", \"acting_verb\": \"lift\", \"purchase_contact\": \"body side\", \"reaction_part\": \"opposite body side\", \"coordination\": \"whole-body\"}",
  "coordination": "whole-body",
  "roles": [
    {
      "id": "A",
      "role": "lift",
      "target": "body",
      "contact_region": "the mid-height of one vertical side wall",
      "function": "provides upward force to raise the box"
    },
    {
   

## Once the prompts feel right
The full batch is just these same functions looped over the whole kept set:
```bash
GEMMA_BACKEND=vllm GEMMA_NO_THINKING=1 VLLM_USE_FLASHINFER_SAMPLER=0 \
  python pipeline/stage0_filter_components.py     # -> outputs/stage0_filtered*.kept.json
GEMMA_BACKEND=vllm GEMMA_NO_THINKING=1 VLLM_USE_FLASHINFER_SAMPLER=0 \
  python pipeline/stage1_task_role_assemble.py    # -> outputs/stage1_redesign_26b.json
```